# Dialogue Summarization: Prompt Engineering Across Model Scale

Compares **zero-shot / one-shot / few-shot** prompting for dialogue
summarization on [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum),
across **FLAN-T5-base** and **FLAN-T5-large**, scored with ROUGE-1/2/L.

Reuses the `src/` package from the companion GitHub repo, so results here
match the numbers reported in the repo's README.

> **Kaggle setup:** enable **Internet** under *Notebook Settings* (needed to
> clone the repo and download the models/dataset from Hugging Face).

In [ ]:
# Install pinned dependencies
!pip install -q transformers==4.40.0 datasets==2.19.0 rouge-score==0.1.2

In [ ]:
# Clone the project repo and make src/ importable.
# Replace the URL below if you rename or fork the repo.
import sys, subprocess, pathlib

REPO_URL = "https://github.com/shimaaelbana/Dialogue-Summarization-Prompt-Engineering-Across-LLM-Models-Scale.git"
REPO_DIR = pathlib.Path("Dialogue-Summarization-Prompt-Engineering-Across-LLM-Models-Scale")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL], check=True)

sys.path.append(str(REPO_DIR.resolve()))

In [ ]:
from src.pipeline import run_experiment

MODEL_NAMES = ["google/flan-t5-base", "google/flan-t5-large"]

# Held-out evaluation indices, disjoint from the few-shot exemplar indices
# ([40, 80, 120]) used inside src/pipeline.py's SHOT_CONFIGS.
TEST_INDICES = [200, 250, 300, 350, 400]

In [ ]:
results_df = run_experiment(
    model_names=MODEL_NAMES,
    test_indices=TEST_INDICES,
)
results_df

In [ ]:
import os

os.makedirs("results", exist_ok=True)
results_df.to_csv("results/results.csv", index=False)
print("Saved results/results.csv")

## Visualize: ROUGE-L by model and shot config

In [ ]:
import matplotlib.pyplot as plt

pivot = results_df.pivot(index="shot_config", columns="model", values="rougeL")
pivot = pivot.reindex(["zero-shot", "one-shot", "few-shot"])

ax = pivot.plot(kind="bar", figsize=(7, 4))
ax.set_ylabel("ROUGE-L (F-measure)")
ax.set_title("Few-shot prompting gain by model scale")
ax.legend(title="Model")
plt.tight_layout()
plt.savefig("results/rougeL_by_shot_config.png", dpi=150)
plt.show()

## Findings

_Fill this in after running the cells above:_

- Does few-shot prompting help `flan-t5-base` as much as it helps
  `flan-t5-large`?
- Where does the ROUGE-L curve start to flatten out as you add exemplars?
- Any qualitative failure cases worth showing (e.g. hallucinated details,
  truncated summaries)?

Paste 2-3 example generations below to make the notebook self-explanatory
for a reader who doesn't want to re-run it.

In [ ]:
# Optional: show a couple of qualitative examples side by side.
from src.data import load_dialogsum
from src.generate import summarize
from src.prompting import build_few_shot_prompt

dataset = load_dialogsum()
example_idx = TEST_INDICES[0]
dialogue = dataset["test"][example_idx]["dialogue"]
reference = dataset["test"][example_idx]["summary"]

for model_name in MODEL_NAMES:
    prompt = build_few_shot_prompt(dataset, "test", [40, 80, 120], dialogue)
    prediction = summarize(prompt, model_name)
    print(f"=== {model_name} (few-shot) ===")
    print("Reference:", reference)
    print("Prediction:", prediction)
    print()